## Exercise: Time Series Forecasting with Deep Learning

This notebook introduces time series forecasting using deep learning and recurrent neural networks (RNNs) with Keras.

You will work with a real-world weather dataset and build models that predict future temperature values.

The notebook is divided into:

- A guided demonstration using a simple RNN
- Extensions where you experiment with LSTM and GRU models
- Exercises and theory questions

**Learning Objectives**

By the end of this exercise, you should be able to:

- Prepare time series data for supervised learning
- Build and train RNN-based forecasting models in Keras
- Perform single-step and multi-step forecasting
- Compare SimpleRNN, LSTM, and GRU architectures
- Reflect on model behavior and theoretical concepts

## 1. Dataset: Weather Time Series

We use the Jena Climate Dataset, a commonly used benchmark dataset for time series forecasting.

It contains 10-minute interval measurements such as:
- Temperature
- Humidity
- Air pressure
- Wind speed

For simplicity, we start by predicting temperature only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
# Downloaded CSV from Jena Climate dataset
url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
df = pd.read_csv(url)

print(df.head())
print(f"\nShape: {df.shape}")
print(f"Columns: {list(df.columns)}")

plt.figure(figsize=(10, 4))
plt.plot(df['T (degC)'])
plt.title("Temperature over time")
plt.xlabel('x10 [minutes]')
plt.ylabel('Temp (degC)')
plt.show()

### Questions

- Which features are available in the dataset?
- Which ones might be useful for forecasting temperature?

**Answer:**

The dataset contains 14 features: Date Time, p (mbar), T (degC), Tpot (K), Tdew (degC), 
rh (%), VPmax (mbar), VPact (mbar), VPdef (mbar), sh (g/kg), H2OC (mmol/mol), rho (g/m**3), 
wv (m/s), max. wv (m/s), wd (deg).

For temperature forecasting, the most useful features would be:
- **T (degC)**: The target variable itself (autoregressive component)
- **p (mbar)**: Atmospheric pressure correlates strongly with temperature changes
- **rh (%)**: Relative humidity is inversely related to temperature
- **Tdew (degC)**: Dew point temperature is physically related to air temperature
- **rho (g/m^3)**: Air density varies with temperature

For this exercise we use only temperature (univariate), which is sufficient to demonstrate 
the RNN approach. Multivariate forecasting would use multiple columns as input features.

## 2. Prepare Sequences

- Select the temperature column
- Normalize values
- Create supervised learning sequences

In [ ]:
temperature = df['T (degC)'].values.reshape(-1, 1)

scaler = MinMaxScaler()
temperature_scaled = scaler.fit_transform(temperature)

def create_sequences(data, seq_len=24):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

SEQ_LEN = 24  # last 24 time steps
X, y = create_sequences(temperature_scaled, SEQ_LEN)

split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")

## 3. Simple RNN Forecasting Model (Single-Step)

In [ ]:
model_rnn = Sequential([
    SimpleRNN(32, activation='tanh', input_shape=(SEQ_LEN, 1)),
    Dense(1)
])

model_rnn.compile(optimizer='adam', loss='mse')
model_rnn.summary()

## 4. Train and Evaluate

In [ ]:
results = []

history_rnn = model_rnn.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_val, y_val),
    batch_size=64
)

In [ ]:
plt.plot(history_rnn.history['loss'], label='Train')
plt.plot(history_rnn.history['val_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid()
plt.title('SimpleRNN Training Curves')
plt.show()

## 5. Evaluate model on validation dataset

In [ ]:
def evaluate_model(model, X_val, y_val, scaler, model_name):
    """Evaluate a model and return metrics in original scale."""
    preds = model.predict(X_val)
    y_true = scaler.inverse_transform(y_val)
    y_pred = scaler.inverse_transform(preds)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f"{model_name} - RMSE: {rmse:.4f} degC, MAE: {mae:.4f} degC")
    return {'model': model_name, 'rmse': rmse, 'mae': mae, 'y_pred': y_pred, 'y_true': y_true}

rnn_result = evaluate_model(model_rnn, X_val, y_val, scaler, 'SimpleRNN')
results.append(rnn_result)

## 6. Single-Step Forecasting

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(rnn_result['y_true'][:400], label='True')
plt.plot(rnn_result['y_pred'][:400], label='Predicted')
plt.legend()
plt.title('Single-Step Forecast (SimpleRNN)')
plt.xlabel('Time step')
plt.ylabel('Temperature (degC)')
plt.grid(True)
plt.show()

## 7. Multi-Step Forecasting

We now predict multiple future time steps recursively.

In [ ]:
def multi_step_forecast(model, start_seq, steps=24):
    """Recursive multi-step forecasting: use each prediction as input for the next."""
    current_seq = start_seq.copy()
    preds = []
    for _ in range(steps):
        next_val = model.predict(current_seq[np.newaxis, ...], verbose=0)[0]
        preds.append(next_val)
        current_seq = np.roll(current_seq, -1, axis=0)
        current_seq[-1] = next_val
    return np.array(preds)

In [ ]:
FORECAST_STEPS = 48

preds_rnn = multi_step_forecast(model_rnn, X_val[0], steps=FORECAST_STEPS)

plt.figure(figsize=(10, 4))
plt.plot(scaler.inverse_transform(y_val[:FORECAST_STEPS]), label='True')
plt.plot(scaler.inverse_transform(preds_rnn), label='Predicted (SimpleRNN)')
plt.legend()
plt.title(f'Multi-Step Forecast ({FORECAST_STEPS} steps, SimpleRNN)')
plt.xlabel('Time step')
plt.ylabel('Temperature (degC)')
plt.grid(True)
plt.show()

---
## 8. Extend model with LSTM and GRU (Task - SOLUTION)

Replace the SimpleRNN layer with:
- LSTM
- GRU

Compare:
- Convergence speed
- Validation loss
- Forecast stability

### 8.1 LSTM Model

LSTM (Long Short-Term Memory) adds a cell state with gating mechanisms that allow it to 
learn long-range dependencies. As described in HOML Ch. 15, the LSTM cell has three gates: 
forget gate, input gate, and output gate (see PDF slides 18-20).

In [ ]:
model_lstm = Sequential([
    LSTM(32, activation='tanh', input_shape=(SEQ_LEN, 1)),
    Dense(1)
])

model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.summary()
print(f"\nLSTM parameters: {model_lstm.count_params()}")
print(f"SimpleRNN parameters: {model_rnn.count_params()}")
print(f"Ratio: {model_lstm.count_params() / model_rnn.count_params():.1f}x")

In [ ]:
history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_val, y_val),
    batch_size=64
)

lstm_result = evaluate_model(model_lstm, X_val, y_val, scaler, 'LSTM')
results.append(lstm_result)

### 8.2 GRU Model

GRU (Gated Recurrent Unit) is a simplified version of LSTM with only two gates 
(reset and update) instead of three. As noted in the slides (page 26), GRU is 
faster to train and performs comparably on simpler tasks.

In [ ]:
model_gru = Sequential([
    GRU(32, activation='tanh', input_shape=(SEQ_LEN, 1)),
    Dense(1)
])

model_gru.compile(optimizer='adam', loss='mse')
model_gru.summary()
print(f"\nGRU parameters: {model_gru.count_params()}")

In [ ]:
history_gru = model_gru.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_val, y_val),
    batch_size=64
)

gru_result = evaluate_model(model_gru, X_val, y_val, scaler, 'GRU')
results.append(gru_result)

### 8.3 Comparison: Convergence Speed

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss comparison
axes[0].plot(history_rnn.history['loss'], label='SimpleRNN', marker='o', markersize=4)
axes[0].plot(history_lstm.history['loss'], label='LSTM', marker='s', markersize=4)
axes[0].plot(history_gru.history['loss'], label='GRU', marker='^', markersize=4)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(True)

# Validation loss comparison
axes[1].plot(history_rnn.history['val_loss'], label='SimpleRNN', marker='o', markersize=4)
axes[1].plot(history_lstm.history['val_loss'], label='LSTM', marker='s', markersize=4)
axes[1].plot(history_gru.history['val_loss'], label='GRU', marker='^', markersize=4)
axes[1].set_title('Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### 8.4 Comparison: Single-Step Predictions

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

plot_range = 400
for ax, res in zip(axes, results):
    ax.plot(res['y_true'][:plot_range], label='True', alpha=0.8)
    ax.plot(res['y_pred'][:plot_range], label='Predicted', alpha=0.8)
    ax.set_title(f"{res['model']} (RMSE={res['rmse']:.4f}, MAE={res['mae']:.4f})")
    ax.legend()
    ax.grid(True)

axes[-1].set_xlabel('Time step')
fig.suptitle('Single-Step Forecast Comparison', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 8.5 Comparison: Multi-Step Forecast Stability

In [ ]:
FORECAST_STEPS = 48

preds_rnn_ms = multi_step_forecast(model_rnn, X_val[0], steps=FORECAST_STEPS)
preds_lstm_ms = multi_step_forecast(model_lstm, X_val[0], steps=FORECAST_STEPS)
preds_gru_ms = multi_step_forecast(model_gru, X_val[0], steps=FORECAST_STEPS)

true_vals = scaler.inverse_transform(y_val[:FORECAST_STEPS])

plt.figure(figsize=(12, 5))
plt.plot(true_vals, label='True', linewidth=2, color='black')
plt.plot(scaler.inverse_transform(preds_rnn_ms), label='SimpleRNN', linestyle='--')
plt.plot(scaler.inverse_transform(preds_lstm_ms), label='LSTM', linestyle='--')
plt.plot(scaler.inverse_transform(preds_gru_ms), label='GRU', linestyle='--')
plt.title(f'Multi-Step Forecast Comparison ({FORECAST_STEPS} steps)')
plt.xlabel('Time step')
plt.ylabel('Temperature (degC)')
plt.legend()
plt.grid(True)
plt.show()

# Multi-step RMSE comparison
for name, preds in [('SimpleRNN', preds_rnn_ms), ('LSTM', preds_lstm_ms), ('GRU', preds_gru_ms)]:
    ms_rmse = np.sqrt(mean_squared_error(
        scaler.inverse_transform(y_val[:FORECAST_STEPS]),
        scaler.inverse_transform(preds)
    ))
    print(f"{name} multi-step RMSE: {ms_rmse:.4f} degC")

### 8.6 Summary Table

In [ ]:
param_counts = {
    'SimpleRNN': model_rnn.count_params(),
    'LSTM': model_lstm.count_params(),
    'GRU': model_gru.count_params()
}

final_val_losses = {
    'SimpleRNN': history_rnn.history['val_loss'][-1],
    'LSTM': history_lstm.history['val_loss'][-1],
    'GRU': history_gru.history['val_loss'][-1]
}

print(f"{'Model':<12} {'Params':>8} {'Val Loss':>12} {'RMSE (degC)':>14} {'MAE (degC)':>14}")
print("-" * 65)
for r in results:
    name = r['model']
    print(f"{name:<12} {param_counts[name]:>8} {final_val_losses[name]:>12.6f} {r['rmse']:>14.4f} {r['mae']:>14.4f}")

---
## 9. Questions and Reflection (ANSWERS)
---

### Q8.a) Why are RNNs suitable for time series forecasting?

RNNs are suitable for time series forecasting because they are specifically designed to 
process sequential data. As described in HOML Chapter 15:

1. **Temporal memory**: RNNs maintain a hidden state that is updated at each time step, 
   allowing them to carry information from earlier time steps to later ones. This is 
   essential for time series where past values influence future values.

2. **Variable-length input**: RNNs can process sequences of any length through the same 
   learned weights, unlike fully connected networks which require fixed input dimensions.

3. **Parameter sharing**: The same weight matrices are applied at every time step. This 
   means the network learns temporal patterns that are translation-invariant in time 
   (a pattern occurring at step 5 is recognized the same way as at step 50).

4. **Natural ordering**: Unlike feed-forward networks, RNNs respect the sequential order 
   of the data, which is fundamental to time series (the order of observations matters).

The recurrent neuron (slide 13) receives both the current input x(t) and the previous 
hidden state h(t-1), computing: h(t) = tanh(W_x * x(t) + W_h * h(t-1) + b)

### Q8.b) What are the limitations of RNN-based forecasting?

1. **Vanishing/Exploding Gradients**: The most fundamental limitation of basic (SimpleRNN) 
   networks. During backpropagation through time (BPTT), gradients are multiplied by the 
   recurrent weight matrix at each step. Over many steps, gradients either shrink to near 
   zero (vanishing) or grow explosively (exploding), making it difficult to learn long-range 
   dependencies (HOML Ch. 15). This is exactly what LSTM was designed to solve.

2. **Sequential computation**: RNNs process one time step at a time, which cannot be 
   parallelized. This makes training slower than architectures like Transformers that 
   process all time steps simultaneously.

3. **Short-term memory** (for SimpleRNN): In practice, SimpleRNN typically only captures 
   dependencies spanning ~10-20 time steps, which may be insufficient for patterns with 
   longer periodicity (e.g., seasonal temperature patterns).

4. **Error accumulation in multi-step forecasting**: When predicting multiple steps ahead 
   recursively, each prediction error is fed back as input, causing errors to compound 
   over time (addressed in Q8.d).

5. **Difficulty with very long sequences**: Even LSTM/GRU struggle with very long sequences 
   (thousands of steps). The hidden state has finite capacity to compress history.

### Q8.c) What problem does LSTM solve compared to SimpleRNN?

LSTM solves the **vanishing gradient problem** that plagues SimpleRNN. 

In a SimpleRNN, the hidden state is overwritten at each time step through a tanh activation. 
During backpropagation, gradients must flow through many such transformations, and the 
repeated multiplication by values < 1 (from tanh derivatives) causes gradients to vanish 
exponentially. This means SimpleRNN cannot learn from events that happened many time steps 
ago.

LSTM introduces a **cell state** (long-term memory) that runs through the entire sequence 
with only linear transformations (additions and element-wise multiplications). This creates 
a "highway" for gradients to flow backward without vanishing. Three gates control this:

1. **Forget gate** (f_t): Decides what to remove from cell state 
   (slide 19: f_t = sigmoid(W_f * [h_{t-1}, x_t] + b_f))

2. **Input gate** (i_t): Decides what new information to store 
   (slide 19: i_t = sigmoid(W_i * [h_{t-1}, x_t] + b_i))

3. **Output gate** (o_t): Decides what to output as hidden state 
   (slide 20: o_t = sigmoid(W_o * [h_{t-1}, x_t] + b_o))

This allows LSTM to selectively remember or forget information over hundreds of time steps, 
making it far more effective for capturing long-range temporal dependencies (HOML Ch. 15).

The trade-off: LSTM has ~4x the parameters of SimpleRNN (4 weight matrices instead of 1) 
and is correspondingly slower to train.

### Q8.d) What causes error accumulation in multi-step forecasting?

In recursive (autoregressive) multi-step forecasting, we predict one step ahead, then feed 
that prediction back as input to predict the next step, and so on. Error accumulation occurs 
because:

1. **Each prediction has some error**: No model is perfect. Even a small error 
   (e.g., 0.2 degC off) exists in every single-step prediction.

2. **Errors compound through feedback**: When we feed a slightly wrong prediction as input 
   for the next step, the model is now operating on data it has never seen during training 
   (the training data was always ground-truth sequences). This causes the next prediction 
   to be even less accurate.

3. **Distribution shift**: Over many recursive steps, the input sequence increasingly 
   consists of model predictions rather than real data. The distribution of these synthetic 
   inputs differs from the training distribution, causing the model to drift further from 
   reality with each step.

This is visible in our multi-step forecast plots: the predicted curve tracks the true values 
well for the first ~5-10 steps, then gradually diverges. The forecast often converges toward 
the mean temperature, losing the dynamic variation of the real signal.

Mitigation strategies (HOML Ch. 15):
- Train a direct multi-output model (predict all future steps at once)
- Use teacher forcing during training
- Sequence-to-sequence architectures with attention

### Q8.e) How does sequence length affect forecasting performance?

Sequence length (the lookback window) determines how much historical context the model sees 
when making a prediction. Its effect is nuanced:

In [ ]:
# Experiment with different sequence lengths
seq_lengths = [6, 12, 24, 48, 96]
seq_results = []

for sl in seq_lengths:
    print(f"\nTraining with SEQ_LEN = {sl}")
    X_sl, y_sl = create_sequences(temperature_scaled, sl)
    sp = int(0.8 * len(X_sl))
    X_tr, X_vl = X_sl[:sp], X_sl[sp:]
    y_tr, y_vl = y_sl[:sp], y_sl[sp:]

    m = Sequential([
        SimpleRNN(32, activation='tanh', input_shape=(sl, 1)),
        Dense(1)
    ])
    m.compile(optimizer='adam', loss='mse')
    m.fit(X_tr, y_tr, epochs=10, validation_data=(X_vl, y_vl), batch_size=64, verbose=0)

    preds = m.predict(X_vl, verbose=0)
    y_true_inv = scaler.inverse_transform(y_vl)
    y_pred_inv = scaler.inverse_transform(preds)
    rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
    mae = mean_absolute_error(y_true_inv, y_pred_inv)

    seq_results.append({'seq_len': sl, 'rmse': rmse, 'mae': mae})
    print(f"  SEQ_LEN={sl}: RMSE={rmse:.4f}, MAE={mae:.4f}")

# Plot results
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([r['seq_len'] for r in seq_results], [r['rmse'] for r in seq_results], 'o-', label='RMSE')
ax.plot([r['seq_len'] for r in seq_results], [r['mae'] for r in seq_results], 's-', label='MAE')
ax.set_xlabel('Sequence Length')
ax.set_ylabel('Error (degC)')
ax.set_title('Effect of Sequence Length on SimpleRNN Performance')
ax.legend()
ax.grid(True)
plt.show()

**Answer Q8.e):**

- **Too short** (e.g., 6 steps): The model sees too little history to capture meaningful 
  temporal patterns. It lacks the context needed for accurate prediction.

- **Moderate** (e.g., 24 steps): A good balance. The model sees enough history to capture 
  local trends and short-term patterns. For 10-minute interval data, 24 steps = 4 hours 
  of context.

- **Too long** (e.g., 96+ steps): For SimpleRNN, performance may degrade because the 
  vanishing gradient problem makes it hard to propagate useful information across many 
  steps. Training also becomes slower. LSTM/GRU handle longer sequences better.

The optimal sequence length depends on the natural periodicity of the data and the model 
architecture's ability to retain long-range information.

### Q8.f) Which model performed best and why?

In [ ]:
# Final comparison
print(f"{'Model':<12} {'RMSE (degC)':>14} {'MAE (degC)':>14} {'Parameters':>12}")
print("-" * 55)
for r in results:
    name = r['model']
    print(f"{name:<12} {r['rmse']:>14.4f} {r['mae']:>14.4f} {param_counts[name]:>12}")

best = min(results, key=lambda x: x['rmse'])
print(f"\nBest model by RMSE: {best['model']} ({best['rmse']:.4f} degC)")

**Answer Q8.f):**

For this task (univariate temperature forecasting with SEQ_LEN=24), all three models 
typically achieve similar single-step RMSE (~0.2 degC). This is because the task is 
relatively simple: predicting the next 10-minute temperature reading from the last 24 
readings is essentially short-range extrapolation where the most recent values carry 
the most information.

However, differences emerge in:

1. **Multi-step forecast stability**: LSTM and GRU produce more stable multi-step forecasts 
   that track the true signal longer before diverging. SimpleRNN tends to converge to the 
   mean faster.

2. **Convergence speed**: LSTM and GRU typically converge faster (lower loss after fewer 
   epochs) because their gating mechanisms allow more effective gradient flow.

3. **GRU vs LSTM**: GRU often matches or slightly outperforms LSTM on this task while 
   being faster to train (fewer parameters). As stated in the slides (page 26): 
   "GRU performs comparable to LSTMs when data is limited or tasks are simpler."

For more complex tasks (multivariate, longer sequences, stock forecasting), LSTM's 
additional capacity may provide a clearer advantage.

---
### Deliverables

- Completed notebook with experiments (SimpleRNN, LSTM, GRU)
- Plots comparing convergence, single-step, and multi-step forecasts
- Answers to all questions Q8.a through Q8.f
- Sequence length experiment (Q8.e)

### Reflection

This exercise demonstrated the RNN family of models for time series forecasting, following 
HOML Chapter 15. Key takeaways:

- SimpleRNN works for short-range, simple forecasting tasks but struggles with long 
  dependencies due to vanishing gradients.
- LSTM's gating mechanism (forget, input, output gates) effectively addresses the vanishing 
  gradient problem, enabling learning of longer-range patterns.
- GRU offers a simpler alternative with comparable performance on many tasks, using 2 gates 
  instead of 3 and a single hidden state instead of separate cell/hidden states.
- Recursive multi-step forecasting suffers from error accumulation regardless of model 
  architecture; direct multi-output approaches can mitigate this.
- Sequence length is a critical hyperparameter: too short loses context, too long 
  introduces noise and gradient issues (especially for SimpleRNN).